In [1]:
import os
import shutil
import zipfile
from pathlib import Path
import pandas as pd
from tqdm import tqdm

In [ ]:

# root_dir = r'Y:\ZHL\isds\PS\task0812'
root_dir = r'E:\data\202502_signboard\data_annotation\ps_data\task0815'
merge_dir = os.path.join(root_dir, 'merge_dir')
root_folder_id = '1aCwmJMvk9cpq6UchMk6SyZQfkEp2ttqd'
client_secret = r"E:\data\202502_signboard\data_annotation\docs\client_secret.json"
token_path = r'E:\repository\dataset_tools\isds_tool\PS_data\token.json'
SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

slam_root_folder_id = '1xr5uLZNxut3Vfhq-hD_FXUaV4RoI3znx'
gap_num = 3

In [4]:
# os.remove(token_path)

In [9]:
import os
import io
from concurrent.futures import ThreadPoolExecutor
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request


def authenticate_with_google(token_path, client_secret_path):
    creds = None

    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(client_secret_path, SCOPES)
            creds = flow.run_local_server(port=0)
        with open(token_path, 'w') as token_file:
            token_file.write(creds.to_json())

    service = build('drive', 'v3', credentials=creds)
    return service


def download_large_file(service, file_id, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    if os.path.exists(file_path):
        print(f"⚠️ 已存在，跳过: {file_path}")
        return
    print(f"⬇️ Downloading {file_path}")
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(file_path, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"⬇️ Downloading {file_path}: {int(status.progress() * 100)}%")
    print(f"✅ Finished: {file_path}")

def download_folder_recursive(service, folder_id, save_path):
    os.makedirs(save_path, exist_ok=True)
    query = f"'{folder_id}' in parents and trashed = false"
    results = service.files().list(q=query, fields="files(id, name, mimeType)").execute()
    items = results.get('files', [])

    for item in items:
        file_id = item['id']
        file_name = item['name']
        file_mime = item['mimeType']
        full_path = os.path.join(save_path, file_name)

        if file_mime == 'application/vnd.google-apps.folder':
            download_folder_recursive(service, file_id, full_path)
        else:
            download_large_file(service, file_id, full_path)

def download_subfolder_task(folder_obj, root_save_path, token_path, client_secret_path):
    # 每个线程都单独认证，避免多线程共享service导致问题
    service = authenticate_with_google(token_path, client_secret_path)
    folder_id = folder_obj['id']
    folder_name = folder_obj['name']
    target_path = os.path.join(root_save_path, folder_name)
    print(f"\n📁 Starting folder: {folder_name}")
    download_folder_recursive(service, folder_id, target_path)

def download_all_subfolders_parallel(token_path, client_secret_path, root_folder_id, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    # 主线程先获取子文件夹列表
    service = authenticate_with_google(token_path, client_secret_path)
    query = f"'{root_folder_id}' in parents and trashed = false and mimeType = 'application/vnd.google-apps.folder'"
    results = service.files().list(q=query, fields="files(id, name)").execute()
    folders = results.get('files', [])

    print(f"将并发下载 {len(folders)} 个子文件夹...\n")

    with ThreadPoolExecutor(max_workers=len(folders)) as executor:
        for folder in folders:
            executor.submit(download_subfolder_task, folder, save_dir, token_path, client_secret_path)



In [ ]:
download_all_subfolders_parallel(token_path, client_secret, root_folder_id, root_dir)

In [20]:
from img_preprocess import select_img
from deduplication_demo import filter_deduplication

def process_dirs(root_dir):
    sub_dirs = os.listdir(root_dir)
    for idx, sub_name in enumerate(sub_dirs):
        sub_dir = os.path.join(root_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['camera1', 'camera2', 'camera3', 'camera4', 'camera5', 'camera6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, cam_name, 'raw')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                print(f'{image_dir_src} selecting...')
                image_dir_select = image_dir_src+'_select'
                shutil.rmtree(image_dir_select) if os.path.exists(image_dir_select) else None
                select_img(image_dir_src, image_dir_select, gap=gap_num)
                print(f'{image_dir_select} filtering...')
                image_dir_filter = image_dir_src+'_filter'
                shutil.rmtree(image_dir_filter) if os.path.exists(image_dir_filter) else None
                filter_deduplication(image_dir_select, image_dir_filter)
                print(f'{image_dir_filter} done\n')

In [22]:
process_dirs(root_dir)

1355


In [ ]:
def img_merge(input_dir, output_dir):
    sub_dirs = os.listdir(input_dir)
    if 'merge_dir' in sub_dirs:
        sub_dirs.remove('merge_dir')
        shutil.rmtree(os.path.join(input_dir, 'merge_dir'))
    os.makedirs(output_dir, exist_ok=True)
    for sub_name in sub_dirs:
        sub_dir = os.path.join(input_dir, sub_name)
        if not os.path.isdir(sub_dir) or sub_name.startswith('r'):
            continue
        cam_name_list = ['camera1', 'camera2', 'camera3', 'camera4', 'camera5', 'camera6']
        for cam_name in cam_name_list:
            image_dir_src = os.path.join(sub_dir, cam_name, 'raw_filter')
            if not os.path.exists(image_dir_src):
                print(f'{image_dir_src} not exists')
            else:
                img_list = os.listdir(image_dir_src)
                for img_name in tqdm(img_list):
                    img_path_src = os.path.join(image_dir_src, img_name)
                    img_path_dst = os.path.join(output_dir, cam_name+'_'+img_name)
                    shutil.copyfile(img_path_src, img_path_dst)



将并发下载 2 个子文件夹...


📁 Starting folder: ymt--2

📁 Starting folder: ymt--1
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--1\geo_ref_matrix.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--2\segment0\geo_ref_matrix.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--1\geo_ref_matrix.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0725\ymt--1\geo_ref_matrix.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--1\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--2\segment0\geo_ref_matrix.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0725\ymt--2\segment0\geo_ref_matrix.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--2\segment0\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--1\poses_tum_converted.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0725\ymt--1\poses_tum_converted.txt
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--1\ymt--1.pcd
⬇️ Downloading Y:\ZHL\isds\PS\task0725\ymt--2\segment0\poses_tum_converted.txt: 100%
✅ Finished: Y:\ZHL\isds\PS\task0725\ymt--2\segment0\poses_tum_converted

In [ ]:
img_merge(root_dir, merge_dir)

In [ ]:
print(len(os.listdir(merge_dir)))

In [ ]:
import zipfile
import os
